# Unit 6 — Linear Regression

Units 4 and 5 asked whether groups differ. Regression asks a wider question:
how does an outcome change as one or more predictors change, and how well does
a straight line describe that? A continuous predictor gives a rate — so much
more outcome per unit of predictor. A categorical one gives a difference
between groups. One model holds both.

Every part of this notebook uses one dataset: exam results for four thousand
pupils in sixty-five London schools. Five questions, in order:

1. Does reading at 11 predict exam results at 16? *(Model 1)*
2. Do girls and boys with the same reading score differ at 16? *(Model 2)*
3. Does the slope differ by sex? *(Model 3, an interaction)*
4. Do pupils at single-sex schools score differently from the same sex at
   mixed schools, at the same reading score? *(Model 4)*
5. How far can the standard errors trust four thousand pupils who sit in
   sixty-five schools?

Checking a model's assumptions — the residuals, influence, collinearity — is
Unit 7's. This unit fits models and reads them.

In [ ]:
import math9102 as m9
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

m9.use_house_style()

---

## Part 1 — Does reading at 11 predict exam results at 16?

The data were built to show how pupils' results vary between schools. Two
variables carry Model 1:

- `normexam`, the exam score at 16;
- `standlrt`, a reading test score at 11.

Both are standardised: a score of zero is the average pupil, and one unit is
one standard deviation.

In [ ]:
regression = m9.load_regression()
regression.shape

In [ ]:
# No missing values on anything this unit uses: the listwise cost is nothing.
missing = m9.missingness(regression, ["normexam", "standlrt", "girl", "schgend", "school"])
print(missing.attrs["summary"])
m9.describe(regression, ["normexam", "standlrt"])

### Look before you model

In [ ]:
m9.normality_panel(regression["normexam"], label="Exam score at 16 (normexam)");

In [ ]:
m9.normality_panel(regression["standlrt"], label="Reading score at 11 (standlrt)");

The outcome's own shape is not an assumption of regression. What the model
assumes is about its **residuals**, the part of each score the line does not
account for, and that is Unit 7's subject. Look at both variables anyway, to
know what you are predicting and from what: here, two roughly symmetric scores
with no gaps or impossible values.

### Scatter, then *r*

In [ ]:
m9.scatter_with_fit(regression, "standlrt", "normexam",
                    xlabel="Reading score at 11 (standlrt)",
                    ylabel="Exam score at 16 (normexam)")

In [ ]:
correlation = stats.pearsonr(regression["standlrt"], regression["normexam"])
m9.report_correlation(regression, "standlrt", "normexam", correlation,
                      x_label="reading score at 11", y_label="exam score at 16")

A rising cloud, wide at every reading score. The correlation says the two move
together; the regression says by how much.

### Fit Model 1

The formula reads "exam score, predicted from reading score". The outcome goes
on the left of the tilde.

In [ ]:
model1 = smf.ols("normexam ~ standlrt", data=regression).fit()
print(model1.summary())

The summary is long. Four things in it matter here, and the cells below take
them one at a time.

### How good is the model? The mean as a baseline

Without a predictor, your best guess for any pupil is the mean exam score. The
**total** sum of squares measures how far the scores sit from that mean. The
line takes some of that distance away (the **model** sum of squares); what is
left is the **residual** sum of squares.

In [ ]:
ss_total, ss_model, ss_resid = model1.centered_tss, model1.ess, model1.ssr
print(f"total {ss_total:.1f} = model {ss_model:.1f} + residual {ss_resid:.1f}")
print(f"R squared = model / total = {ss_model / ss_total:.3f}")

In [ ]:
# The sums of squares, the mean squares and F, as a table.
sm.stats.anova_lm(model1)

*F* is the model's mean square over the residual mean square: how much the line
accounts for per degree of freedom it uses, against how much is left per degree of
freedom remaining. It tests one null hypothesis — that the line does no better
than the mean.

**Read *R²* as a description of this sample.** It is the share of the variation
in exam scores the line accounts for here. It is not the share of exam success
that reading *causes*, and a sample with a narrower range of readers would give
a smaller *R²* for the same slope.

### Reading the coefficients

In [ ]:
pd.DataFrame({"b": model1.params, "p": model1.pvalues}).join(
    model1.conf_int().set_axis(["CI low", "CI high"], axis=1))

- **The slope** is the difference in predicted exam score between two pupils
  one unit apart in reading score. Its interval says which slopes the data are
  consistent with.
- **The intercept** is the prediction when every predictor is zero — here, for
  a pupil of average reading score. Its *p*-value tests whether that prediction
  is zero, which is rarely a question anyone asked. Do not report it as a
  finding.

### Using the model to predict

In [ ]:
# Predictions for three new pupils: one SD below the mean, at it, above it.
new_pupils = pd.DataFrame({"standlrt": [-1, 0, 1]})
model1.predict(new_pupils)

A prediction is the line's value, not a promise about any one pupil: the cloud
around the line is wide. `get_prediction` gives both kinds of interval.

In [ ]:
model1.get_prediction(new_pupils).summary_frame(alpha=0.05)

`mean_ci_lower` and `mean_ci_upper` bound the **average** exam score of pupils
with that reading score: four thousand pupils pin it down, so the interval is
narrow. `obs_ci_lower` and `obs_ci_upper` bound **one** pupil's score, and are
far wider, because a single pupil carries the residual as well. Both hold only
inside the range of reading scores the model saw.

### Standardised coefficients

Standardising both variables puts the slope in standard-deviation units, so
slopes on different scales can be set side by side. patsy's `standardize()`
does it inside the formula.

In [ ]:
beta = smf.ols("standardize(normexam) ~ standardize(standlrt)", data=regression).fit()
beta.params

These two variables were already standardised in the data, so the slope barely
moves. **Do not standardise a binary predictor.** "One standard deviation of
being a girl" describes nobody; its unstandardised coefficient, the gap between
the groups, is already the readable number.

### Reporting Model 1

In [ ]:
m9.report_regression(
    model1,
    outcome_label="exam score at 16",
    predictor_labels={"standlrt": "reading score at 11"},
)

The paragraph gives *F* with both degrees of freedom, *R²* in words, the
coefficient with its interval, the fitted equation, and what the model does not
establish. With one predictor it says nothing about "holding other factors
constant": there are no other factors in the model to hold.

---

## Part 2 — Do girls and boys with the same reading score differ at 16?

`girl` holds text, "boy" or "girl". The formula turns it into a **dummy**
variable: one for girls, zero for boys. Boys come first alphabetically, so they
are the **reference** category, and the coefficient is the difference between
girls and boys.

In [ ]:
m9.frequency(regression, "girl")

In [ ]:
model2 = smf.ols("normexam ~ standlrt + girl", data=regression).fit()
pd.DataFrame({"Model 1": model1.params, "Model 2": model2.params})

Reading's coefficient moves slightly when `girl` enters. In Model 2 each
coefficient is read **holding the other predictor constant**: the reading slope
for pupils of the same sex, and the girl–boy gap for pupils with the same
reading score.

### Two parallel lines

Model 2 gives boys and girls the same slope and different intercepts. The gap
between the lines is the `girl` coefficient, and because the lines are
parallel it is the same at every reading score.

In [ ]:
lines = pd.DataFrame({"standlrt": [0, 0], "girl": ["boy", "girl"]})
lines["predicted"] = model2.predict(lines)
lines

**Read the gap where you say you read it.** The intercepts are predictions at
`standlrt` = 0, the pupil of average reading score. Evaluate the lines at a
different reading score and the two predictions both move — the gap does not,
but a table of predictions at that score no longer describes "the average
pupil".

### Comparing Model 2 with Model 1

Does adding `girl` improve the model? Two nested models — one the other plus
something — are compared with an *F*-test for the extra terms. **Both must be
fitted on the same rows.** Models fitted on different rows differ in their data
as well as their predictors, and neither *F* nor the change in *R²* can
separate the two. These data have no missing values, so the rows are the same;
the check below compares the rows themselves rather than assuming it.

In [ ]:
assert model1.model.data.row_labels.equals(model2.model.data.row_labels)

# The F-test for the extra term, from statsmodels.
sm.stats.anova_lm(model1, model2)

In [ ]:
# The same test, narrated. It refuses two models fitted on different rows.
m9.report_model_comparison(model1, model2,
                           predictor_labels={"standlrt": "reading score at 11",
                                             "girl[T.girl]": "sex (girl against boy)"})

In [ ]:
m9.report_regression(
    model2,
    outcome_label="exam score at 16",
    predictor_labels={"standlrt": "reading score at 11",
                      "girl[T.girl]": "girl (against boy)"},
)

***R²* never falls when a predictor is added**, even a useless one: the line
can always fit these particular pupils at least as well. Adjusted *R²* charges
for each predictor, which is why the report gives both. A rise in *R²* is
evidence only through its test.

---

## Part 3 — Does the slope differ by sex?

Model 2 forces girls and boys onto one slope. An **interaction** lets the slope
differ: `standlrt * girl` means `standlrt + girl + standlrt:girl`, and the last
term is the difference between the girls' slope and the boys'.

In [ ]:
# The product is built by the formula, so it always matches its parts.
model3 = smf.ols("normexam ~ standlrt * girl", data=regression).fit()
model3.params

Build the interaction in the formula rather than from a column of products.
The dataset carries such a column, `interaction`; it happens to be right, but
a model built from it would not know it was an interaction, and could not be
checked against its parts.

In [ ]:
interaction = "standlrt:girl[T.girl]"
low, high = model3.conf_int().loc[interaction]
print(f"interaction b = {model3.params[interaction]:.3f}, "
      f"95% CI [{low:.3f}, {high:.3f}], p = {model3.pvalues[interaction]:.3f}")

In [ ]:
m9.report_regression(
    model3,
    outcome_label="exam score at 16",
    predictor_labels={"standlrt": "reading score at 11",
                      "girl[T.girl]": "girl (against boy)",
                      interaction: "reading × girl"},
)

**One *p*-value, generated.** The interaction is not statistically significant,
and its interval is narrow and sits across zero.

**What that does not establish.** It does not show the slopes are equal; it
shows that these data cannot tell them apart, and the interval says how
different they could still be. Nor can you claim that sex "moderates" the
reading slope from a model that never let the slope vary — only a model with
the interaction in it can address that question, and this one finds no
evidence for it.

With an interaction in the model, the `girl` coefficient changes meaning: it is
the girl–boy gap where `standlrt` is zero, not at every reading score. The
report says so.

In [ ]:
m9.report_model_comparison(model2, model3,
                           predictor_labels={"standlrt": "reading score at 11",
                                             "girl[T.girl]": "girl (against boy)",
                                             interaction: "the reading × girl interaction"})

---

## Part 4 — Do pupils at single-sex schools score differently?

`schgend` records the type of school: mixed, boys' or girls'. Three categories
need two dummies. Model 4 adds school type to Model 2, and it is tempting to
read each coefficient "holding the others fixed".

**Before any "holding fixed", check the combinations exist.**

In [ ]:
# Before any model: who attends which type of school?
pd.crosstab(regression["schgend"], regression["girl"])

No girl attends a boys' school, and no boy a girls' school. So "a boys' school,
holding sex fixed" cannot be about a girl: there are none to hold. Within a
single-sex school sex never varies, so girls and boys can be compared only in
the mixed schools. The model will print a coefficient either way; the cross-tab
is what tells you which comparisons the data contain.

### School type, read within sex

Enter school type as one categorical predictor beside `girl`, and name the
reference — mixed schools — rather than let the alphabet choose it.

In [ ]:
model4 = smf.ols("normexam ~ standlrt + girl + C(schgend, Treatment('mixedsch'))",
                 data=regression).fit()
model4.params

Because no girl attends a boys' school, each coefficient makes one comparison,
at the same reading score:

- `girl`: girls against boys, in the mixed schools, the only ones with both;
- boys' school: boys there against boys at mixed schools;
- girls' school: girls there against girls at mixed schools.

In [ ]:
# k categories, k - 1 dummies: two school-type terms against the mixed reference.
school_terms = {"girl[T.girl]": "girl (in mixed schools)",
                "C(schgend, Treatment('mixedsch'))[T.boysch]": "boys' school (boys against boys at mixed)",
                "C(schgend, Treatment('mixedsch'))[T.girlsch]": "girls' school (girls against girls at mixed)",
                "standlrt": "reading score at 11"}
m9.report_regression(model4, outcome_label="exam score at 16", predictor_labels=school_terms)

Model 4 is Model 2 with school type added, so the two compare like Models 1
and 2, as one block, on the same rows:

In [ ]:
m9.report_model_comparison(model2, model4, predictor_labels={
    "C(schgend, Treatment('mixedsch'))": "school type",
    "girl[T.girl]": "sex", "standlrt": "reading score at 11"})

**Within sex, and not an effect of.** Each school-type coefficient compares one
sex with the same sex. The model would also predict a score for a girl at a
boys' school, a pupil who does not exist: never report one. And it does not
measure what a school does: families choose schools, and nothing here
separates the school from the pupils who go to it.

---

## Part 5 — How far can the standard errors trust these pupils?

Every interval so far treats the pupils as independent of one another. They are
not: they sit in sixty-five schools, and pupils in one school share teachers,
intake and neighbourhood.

### The intraclass correlation

The **ICC** is the share of the variation in exam scores that lies *between*
schools. The **design effect** says roughly how many times larger the sampling
variance of the average exam score is than it would be for the same number of
unrelated pupils. It is worked out as if every school were the average size,
and these schools are far from that, so read it as a rough guide.

In [ ]:
clustering = m9.icc(regression, "normexam", "school")
clustering

In [ ]:
sizes = regression["school"].value_counts()
print(f"school sizes run from {sizes.min()} to {sizes.max()} pupils")

In [ ]:
n_pupils = len(regression)
print(f"For the average exam score, {n_pupils} pupils carry about as much "
      f"independent information as {n_pupils / clustering['design_effect']:.0f} "
      "unrelated pupils would")

### Cluster-robust standard errors

Cluster-robust standard errors let pupils in one school be alike, and assume
only that schools are independent of one another. The coefficients do not
move; only their standard errors do, and with them the intervals and tests,
which now use *t* on the number of schools rather than the number of pupils.
How much a coefficient loses depends on how much its predictor clusters.

In [ ]:
def by_school(formula):
    return smf.ols(formula, data=regression).fit(
        cov_type="cluster", cov_kwds={"groups": regression["school"]}, use_t=True)

model2_cl = by_school("normexam ~ standlrt + girl")
model4_cl = by_school("normexam ~ standlrt + girl + C(schgend, Treatment('mixedsch'))")

pd.DataFrame({
    "b (unchanged)": model4.params,
    "SE, OLS": model4.bse,
    "SE, clustered": model4_cl.bse,
    "how much wider": model4_cl.bse / model4.bse,
})

The two school-type standard errors widen most, because school type varies
only **between** schools: there are only a few dozen schools to compare,
however many pupils sit in them. Reading's widens by less. And `girl`'s barely
moves: in Model 4 it compares girls and boys inside the same mixed schools, so
whatever a school adds to its pupils cancels out of the comparison. Without
school type, in Model 2, the same coefficient also compares pupils across
schools:

In [ ]:
(model2_cl.bse / model2.bse).rename("Model 2: how much wider")

There, `girl`'s standard error roughly doubles. The same predictor in two
models: what clustering costs a coefficient depends on whether the comparison
it makes is within schools or between them.

In [ ]:
m9.report_regression(model4_cl, outcome_label="exam score at 16",
                     predictor_labels=school_terms)

Read the two school-type intervals in this paragraph against Part 4's. There,
both differences were clearly non-zero. With the clustering acknowledged, both
intervals reach to about zero: the boys'-school interval only just excludes it,
and the girls'-school interval only just includes it. The two differences are
about the same size, so do not report one as real and the other as absent:
read the intervals, not which side of .05 each *p*-value falls. Note too that
*F*'s second degree of freedom has fallen from thousands to one less than the
number of schools: that is how much independent information the test has.

**A repair, not the treatment.** The model built for pupils in schools is a
multilevel model, which is beyond this module. What you can always do is
notice the clustering, and say what it does to your claims.

---

## What to take from this unit

- A regression coefficient is a **rate** for a continuous predictor and a
  **difference** for a categorical one, read holding the model's other
  predictors constant — and only those.
- *F* tests the whole model against the mean; *R²* says how much of the
  variation the model accounts for **in this sample**. It never falls as
  predictors are added; adjusted *R²* charges for them.
- Every coefficient comes with an interval. The intercept is a prediction at
  zero, rarely a finding.
- A categorical predictor becomes *k* − 1 dummies against a reference you
  **name**. Check by cross-tabulation that the combinations a model adjusts
  over exist, and say which comparison each coefficient makes.
- Build an interaction in the formula. A non-significant one shows the data
  cannot tell the slopes apart, not that they are equal.
- Compare nested models **on the same rows**.
- Cluster-robust standard errors leave the coefficients alone and widen their
  standard errors, most for a predictor that varies only between clusters and
  least for a comparison made within them.
- No regression on observational data establishes cause, and *R²* is not a
  share of the reasons.

## Exercise

`exercise.md` asks you to fit four models of sleepiness on the sleep study —
all on one sample — to choose a reference category by counting, and to
criticise a regression write-up.